In [1]:
import torch
from torch import nn
from transformers import BartTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments, BartConfig, AdamW
from tqdm.notebook import tqdm
import pandas as pd
from datasets import Dataset, load_metric, DatasetDict
from torch.nn.init import xavier_uniform_
import torch.nn.functional as F
from transformers import AdamW, get_linear_schedule_with_warmup


def init_params(model):
    for name, param in model.named_parameters():
        if param.data.dim() > 1:
            xavier_uniform_(param.data)
        else:
            pass
model_name = 'facebook/bart-base'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
'''tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
print ("device ",device)
model = model.to(device)'''
mapping = {"N":"negation", "E":"entity swap", "#":"number swap", 
           "A":"to antonym", "I":"no information", 
           "X":"mutual exclusion"}
mapping2 = {"N":0, "E":1, "#":2, 
           "A":3, "I":4, 
           "X":5, "Ans":6}
mapping3 = {0:"N", 1:"E", 2:"#", 
           3:"A", 4:"I", 
           5:"X", 6:"Ans"}  

model_name = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(model_name)
config = BartConfig.from_pretrained(model_name)


class MTLModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Define a classification head
        self.classifier = nn.Linear(config.d_model, len(mapping2))
        self.dropout = nn.Dropout(0.2)
        self.bart = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
        self.criterion = nn.CrossEntropyLoss()
        self.loss_weights = [1, 0.2]
        init_params(self.classifier)
        self.optimizer = None
        self.scheduler = None

    def forward(self, input_ids, attention_mask, labels=None, 
                classification_input_ids=None, classification_attention_mask=None, classification_labels=None):
        # Generation outputs
        outputs = self.bart(input_ids=input_ids, attention_mask=attention_mask, labels=labels, output_hidden_states=True)
        
        # Classification outputs
        classification_outputs = self.bart(
            input_ids=classification_input_ids,
            attention_mask=classification_attention_mask,
            output_hidden_states=True
        )

        # Clone hidden states to avoid shared memory issues
        hidden_states = classification_outputs.decoder_hidden_states[-1].detach().clone()

        # Identify the EOS token
        eos_token_id = self.bart.config.eos_token_id
        eos_mask = classification_input_ids.eq(eos_token_id).detach()  # Ensure the mask is detached

        # Get the EOS hidden state
        eos_hidden_state = hidden_states[eos_mask].view(hidden_states.size(0), -1, hidden_states.size(-1))[:, -1, :].clone()

        eos_hidden_state = self.dropout(eos_hidden_state)
        
        # Apply classification head on the EOS hidden state
        classification_logits = self.classifier(eos_hidden_state)

        if self.training:
            #### Ensure classification_labels has the correct shape
            if classification_labels.dim() > 1:
                classification_labels = classification_labels.squeeze(-1).clone()  # Clone to ensure it's a separate tensor
            #### NEW
            # Convert classification_logits and classification_labels to the same floating point type
            classification_labels = classification_labels.long()
            
            # Calculate generation and classification loss
            loss_g = outputs.loss
            loss_r = self.criterion(classification_logits, classification_labels)
            ### NEW
            loss_r = loss_r.float()  # Ensure the loss is a floating-point type
            weighted_loss_g = self.loss_weights[0] * loss_g
            weighted_loss_r = self.loss_weights[1] * loss_r
            
            return weighted_loss_g + weighted_loss_r, classification_logits, loss_g

        # Modify input for generation using the classification result
        predicted_class = classification_logits.argmax(dim=-1).detach()  # Detach to ensure it does not affect gradient calculation
        modified_decoder_input_ids = self.modify_decoder_input(input_ids, predicted_class)

        # Generate text using modified input
        generation_outputs = self.bart.generate(input_ids=modified_decoder_input_ids, attention_mask=attention_mask)

        return generation_outputs, classification_logits

    def modify_decoder_input(self, input_ids, predicted_class):
        # Modify the input_ids based on the predicted class
        predicted_class_ids = predicted_class.unsqueeze(1).clone()  # Clone to ensure it's a separate tensor
        modified_input_ids = torch.cat([predicted_class_ids, input_ids[:, 1:].clone()], dim=1)  # Clone slices
        return modified_input_ids
    
    def configure_optimizers(self, learning_rate=5e-5, weight_decay=0.01, adam_epsilon=1e-8, warmup_steps=0, total_steps=0):
        no_decay = ["bias", "LayerNorm.weight"]
        optimizer_grouped_parameters = [
            {
                "params": [
                    p for n, p in self.named_parameters()
                    if not any(nd in n for nd in no_decay)
                ],
                "weight_decay": weight_decay
            },
            {
                "params": [
                    p for n, p in self.named_parameters()
                    if any(nd in n for nd in no_decay)
                ],
                "weight_decay": 0.0
            }
        ]
        optimizer = AdamW(
            optimizer_grouped_parameters,
            lr=learning_rate,
            eps=adam_epsilon
        )
        self.optimizer = optimizer
        
        # Create the learning rate scheduler
        if warmup_steps > 0 and total_steps > 0:
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps
            )
        else:
            scheduler = None
        self.scheduler = scheduler
        
        return optimizer, scheduler

    def optimizer_step(self, epoch, batch_idx):
        if self.optimizer is not None:
            self.optimizer.step()
            self.optimizer.zero_grad()
            if self.scheduler is not None:
                self.scheduler.step()



#model = MTLModel.from_pretrained('./fine-tuned-bartMTL-salience') replace these names with the checkpoint....
#model.load_state_dict(torch.load('./fine-tuned-bartMTL-salience/pytorch_model.bin'))
model = MTLModel(config).to(device)

2024-08-21 10:18:50.262403: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-21 10:18:50.300422: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-21 10:18:51.062595: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/hmoradis/.conda/envs/QA_env/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(

In [2]:
def mission(context, question, label, sal):
    if label != "Ans":
        st = f"""Context: {context}

        Question: {question}

        Issue: {mapping[label]}

        Salient Sentence: {sal}

        Task: Generate a new question based on the context and salient sentence that can be answered with the given context, given the issue."""
    else:
        st = question
    return st

def labelLess(context, question, sal):
    st = f"""Context: {context}

    Question: {question}

    Salient Sentence: {sal}
    """
    return st



def tokenize_function(examples):
    
    classification_inputs = [
        labelLess(context if label != "I" else noinfoc, question if label != "I" else noinfoq, sal)
        for context, question, label, sal, noinfoc, noinfoq in zip(examples['context'], examples['potential_UQ_question'], examples['label'], examples['answer_sentence'], examples['old_context'], examples['original_question'])
    ]
    classification_targets = [
        mapping2[label] for label in examples['label']
    ]
    
    # Prepare inputs
    inputs = [
        mission(context if label != "I" else noinfoc, question if label != "I" else noinfoq, label, sal)
        for context, question, label, sal, noinfoc, noinfoq in zip(examples['context'], examples['potential_UQ_question'], examples['label'], examples['answer_sentence'], examples['old_context'], examples['original_question'])
    ]
    targets = [
        original_q if label != "I" else potential_q 
        for label, original_q, potential_q in zip(examples['label'], examples['original_question'], examples['potential_UQ_question'])
    ]
    
    # Tokenize inputs
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length')
    
    # Tokenize targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True, padding='max_length')
    
    # Ensure the labels are correctly formatted
    labels_ids = labels['input_ids']
    # Align the lengths of labels and inputs
    
    classification_inputs = tokenizer(classification_inputs, max_length=512, truncation=True, padding='max_length')
    

    return {
        'input_ids': model_inputs['input_ids'],
        'attention_mask': model_inputs['attention_mask'],
        'labels': labels_ids,
        'classification_input_ids': classification_inputs['input_ids'],
        'classification_attention_mask': classification_inputs['attention_mask'],
        'classification_labels': classification_targets
    }

In [3]:
csv_file = 'all_unans_salience.csv'  # Replace with your CSV file path
df = pd.read_csv(csv_file)
df

,Unnamed: 0,context,original_question,potential_UQ_question,original_answer_span,answer_sentence,label,old_context,label_indicator
0,0,"In 1609, while still there, Smyth wrote a trac...",Smyth believed a scriptural church should cons...,Smyth believed a scriptural church should cons...,baptized on a personal confession of faith,"In 1609, while still there, Smyth wrote a trac...",A,NaN,0
1,1,"In 1609, while still there, Smyth wrote a trac...",Smyth believed a scriptural church should cons...,Smyth believed a scriptural church should cons...,baptized on a personal confession of faith,"In 1609, while still there, Smyth wrote a trac...",A,NaN,0
2,2,"In 1609, while still there, Smyth wrote a trac...",Smyth believed a scriptural church should cons...,Smyth believed a scriptural church should cons...,baptized on a personal confession of faith,"In 1609, while still there, Smyth wrote a trac...",A,NaN,0
3,3,"In October 2015, TCM announced the launch of t...",How long do TCM Wineclub subscriptions last?,How unretentive do TCM Wineclub subscriptions ...,3 month,"Wines are available in 3 month subscriptions, ...",A,NaN,0
4,4,Some Christian writers considered the possibil...,As what was the event mistaken by some pagans?,As what was the event mistaken by no pagans?,a solar eclipse,Some Christian writers considered the possibil...,A,NaN,0
...,...,...,...,...,...,...,...,...,...
20471,20471,Most residents belong to the Anglican Communio...,The Diocese of Saint Helena has it's own what?,The Diocese of Saint Helena has it's own what?,bishop,Most residents belong to the Anglican Communio...,Ans,NaN,1
20472,20472,Other Christian denominations on the island in...,What year did the Salvation Army show up on Sa...,What year did the Salvation Army show up on Sa...,1884,Other Christian denominations on the island in...,Ans,NaN,1
20473,20473,Other Christian denominations on the island in...,What year did the Salvation Army show up on Sa...,What year did the Salvation Army show up on Sa...,1884,Other Christian denominations on the island in...,Ans,NaN,1
20474,20474,Other Christian denominations on the island in...,When did Baptists come to the island?,When did Baptists come to the island?,1845,Other Christian denominations on the island in...,Ans,NaN,1


In [4]:
# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)
# Split the dataset into train and validation sets
dataset = dataset.train_test_split(test_size=0.2)
dataset = DatasetDict({"train": dataset["train"], "validation": dataset["test"]})
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/16380 [00:00<?, ? examples/s]

/home/hmoradis/.conda/envs/QA_env/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:4126: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/4096 [00:00<?, ? examples/s]

In [5]:
def filter_dict(example):
    return {
        'input_ids': example['input_ids'],
        'attention_mask': example['attention_mask'],
        'labels': example['labels'],
        'classification_input_ids': example['classification_input_ids'],
        'classification_attention_mask': example['classification_attention_mask'],
        'classification_labels': example['classification_labels']
    }

filtered_dataset_dict = {}
for split, dataset in tokenized_datasets.items():
    filtered_dataset_dict[split] = dataset.map(
        filter_dict, 
        remove_columns=[col for col in dataset.column_names if col not in ['input_ids', 'attention_mask', 'labels','classification_input_ids','classification_attention_mask', 'classification_labels']]
    )

Map:   0%|          | 0/16380 [00:00<?, ? examples/s]

Map:   0%|          | 0/4096 [00:00<?, ? examples/s]

In [6]:
train = filtered_dataset_dict['train']
validation = filtered_dataset_dict['validation']

In [7]:
for k, v in train[0].items():
    print(k)

input_ids
attention_mask
labels
classification_input_ids
classification_attention_mask
classification_labels


In [8]:
#from safetensors.torch import load_file
'''output_dir = './checkpoint-44862'
#output_dir = './fine-tuned-bartMTL-salience'

# Load the configuration
config = BartConfig.from_pretrained(output_dir)

# Reconstruct the model
model = MTLModel(config)
model.bart.model.encoder.load_state_dict(load_file(os.path.join(output_dir, "encoder.safetensors")))
model.bart.model.decoder.load_state_dict(load_file(os.path.join(output_dir, "decoder.safetensors")))
model.bart.model.shared.load_state_dict(load_file(os.path.join(output_dir, "shared.safetensors")))
model.bart.lm_head.load_state_dict(load_file(os.path.join(output_dir, "lm_head.safetensors")))
model.classifier.load_state_dict(load_file(os.path.join(output_dir, "classifier.safetensors")))

# Load the tokenizer
tokenizer = BartTokenizer.from_pretrained(output_dir)

# Now you can use `model` and `tokenizer` as needed
model.to(device)'''
model.train()
from transformers import Trainer
import os
from safetensors.torch import save_file

class CustomMTLTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        # Extract necessary inputs from the batch
        input_ids = inputs['input_ids']
        attention_mask = inputs['attention_mask']
        labels = inputs.get('labels', None)
        classification_input_ids = inputs.get('classification_input_ids', None)
        classification_attention_mask = inputs.get('classification_attention_mask', None)
        classification_labels = inputs.get('classification_labels', None)
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            classification_input_ids=classification_input_ids,
            classification_attention_mask=classification_attention_mask,
            classification_labels=classification_labels,
        )
        
        # Get the loss
        loss = outputs[0]  # Assuming your model's forward method returns the loss as the first output
        ### NEW
         # Ensure the loss is a floating-point type
        loss = loss.float().mean()  # Convert loss to float and calculate the mean

        return (loss, outputs) if return_outputs else loss

    def save_model(self, output_dir=None, _internal_call=False):
        # If output_dir is None, use self.args.output_dir
        if output_dir is None:
            output_dir = self.args.output_dir
        
        os.makedirs(output_dir, exist_ok=True)

        # Save each component of the model separately
        save_file(self.model.bart.model.encoder.state_dict(), os.path.join(output_dir, "encoder.safetensors"))
        save_file(self.model.bart.model.decoder.state_dict(), os.path.join(output_dir, "decoder.safetensors"))
        save_file(self.model.bart.model.shared.state_dict(), os.path.join(output_dir, "shared.safetensors"))
        save_file(self.model.bart.lm_head.state_dict(), os.path.join(output_dir, "lm_head.safetensors"))
        
        # Save the classifier separately
        save_file(self.model.classifier.state_dict(), os.path.join(output_dir, "classifier.safetensors"))

        # Save tokenizer, config, and other necessary files
        self.tokenizer.save_pretrained(output_dir)
        self.model.bart.config.save_pretrained(output_dir)



training_args = TrainingArguments(
    output_dir='./results-mtl-run',  # Specify where to save checkpoints and outputs
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=6,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy='epoch',
    save_steps=1000,
    save_total_limit=2
)

# Initialize the custom trainer
trainer = CustomMTLTrainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=validation,
    tokenizer=tokenizer  # Include tokenizer if you need it for evaluation
)



# Train the model
#epoch_callback = SaveModelEveryEpochCallback(save_path='./results-mtl')

#trainer.add_callback(epoch_callback)

# Train the model
trainer.train()   

# checkpoint_path = "./results-mtl"  # Just the root directory, not a specific checkpoint
# trainer.train(resume_from_checkpoint=checkpoint_path)
# Save the model


/home/hmoradis/.conda/envs/QA_env/lib/python3.11/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: julien-serbanescu (juteam). Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.537200,2425.041748
2,0.528100,2413.926758
3,0.556800,2384.605957
4,0.489700,2397.411133
5,0.491400,2399.702637
6,0.463300,2391.830566


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file

TrainOutput(global_step=24570, training_loss=0.5632324933520555, metrics={'train_runtime': 7729.6498, 'train_samples_per_second': 12.715, 'train_steps_per_second': 3.179, 'total_flos': 0.0, 'train_loss': 0.5632324933520555, 'epoch': 6.0})

# Load from checkpoints

In [ ]:
import os

# List all checkpoint directories
checkpoint_dir = "./results-mtl"
checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint-")]

print("Available checkpoints:", checkpoints)

# Check the contents of a specific checkpoint
checkpoint_path = os.path.join(checkpoint_dir, "checkpoint-22000")
print("Files in checkpoint-22000:", os.listdir(checkpoint_path))


In [ ]:
from safetensors.torch import load_file
import torch

# Load model components manually
model.bart.model.encoder.load_state_dict(load_file('./results-mtl/checkpoint-22000/encoder.safetensors'))
model.bart.model.decoder.load_state_dict(load_file('./results-mtl/checkpoint-22000/decoder.safetensors'))
model.bart.model.shared.load_state_dict(load_file('./results-mtl/checkpoint-22000/shared.safetensors'))
model.bart.lm_head.load_state_dict(load_file('./results-mtl/checkpoint-22000/lm_head.safetensors'))
model.classifier.load_state_dict(load_file('./results-mtl/checkpoint-22000/classifier.safetensors'))

# Load optimizer and scheduler state
optimizer_state = torch.load('./results-mtl/checkpoint-22000/optimizer.pt')
scheduler_state = torch.load('./results-mtl/checkpoint-22000/scheduler.pt')

# Optionally, load RNG state for reproducibility
rng_state = torch.load('./results-mtl/checkpoint-22000/rng_state.pth')

# Initialize the trainer
trainer = CustomMTLTrainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=validation,
    tokenizer=tokenizer,
#     optimizers=(optimizer, scheduler)  # Reassign optimizer and scheduler
)

# Continue training
trainer.train()


In [9]:
output_dir = './fine-tuned-bartMTL-salience'
os.makedirs(output_dir, exist_ok=True)

# Save each component of the model separately
save_file(model.bart.model.encoder.state_dict(), os.path.join(output_dir, "encoder.safetensors"))
save_file(model.bart.model.decoder.state_dict(), os.path.join(output_dir, "decoder.safetensors"))
save_file(model.bart.model.shared.state_dict(), os.path.join(output_dir, "shared.safetensors"))
save_file(model.bart.lm_head.state_dict(), os.path.join(output_dir, "lm_head.safetensors"))

# Save the classifier separately
save_file(model.classifier.state_dict(), os.path.join(output_dir, "classifier.safetensors"))

# Save tokenizer and config
tokenizer.save_pretrained(output_dir)
model.bart.config.save_pretrained(output_dir)

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
